# Fundamentals 10 - Environment Eval API

**Fundamentals explora la API.** Este notebook muestra environments y evals sobre un batch de **5 problemas multi-operación**.

No hay parser de lenguaje natural ni módulo aritmético agrupado. El batch se escribe explícito para que el usuario vea qué recibe `AgenticEnvironment` y qué valida `run_eval`.


In [ ]:
from typing import Any

import agentic_systems as lab

lab.show({
    "checkpoint": "2.4.9.9",
    "layer": "fundamentals",
    "focus": "explorar Environment + Eval API con definiciones visibles",
    "environment_design": "Gymnasium-like without Gymnasium dependency",
})


## Problema default de fundamentals

Todos los notebooks de `tutorials/` usan este mismo problema para comparar la API sin cambiar de caso cada vez.


In [ ]:
USER_PROMPT = """Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2.
Dime:
- procedimiento
- resultado final""".strip()

# La estructura solicitada por el usuario se materializa como datos simples.
# No es un parser ni una respuesta precocinada: sólo representa la sección `Dime:`.
REQUESTED_OUTPUTS = ["procedimiento", "resultado_final"]

lab.show({
    "prompt_usuario": USER_PROMPT,
    "salidas_solicitadas": REQUESTED_OUTPUTS,
}, title="Problema default · visible")


## 1) Batch de 5 casos

Cada fila es un step del episodio. El primer caso es exactamente el problema default; los otros cuatro tienen entre 3 y 5 operaciones.


In [ ]:
def make_math_batch() -> list[dict[str, Any]]:
    """Devuelve 5 problemas; cada registro muestra prompt, operaciones y salidas solicitadas."""

    return [
        {
            "case_id": "math_01_default",
            "prompt": USER_PROMPT,
            "initial": 10,
            "operations": [("add", 20), ("subtract", 9), ("multiply", 4), ("divide", 2)],
            "requested_outputs": REQUESTED_OUTPUTS,
        },
        {
            "case_id": "math_02",
            "prompt": """Empieza con 100, resta 25, divide entre 5 y suma 7.
Dime:
- procedimiento
- resultado final""".strip(),
            "initial": 100,
            "operations": [("subtract", 25), ("divide", 5), ("add", 7)],
            "requested_outputs": REQUESTED_OUTPUTS,
        },
        {
            "case_id": "math_03",
            "prompt": """Empieza con 7, multiplica por 6, suma 8, resta 10 y divide entre 5.
Dime:
- procedimiento
- resultado final""".strip(),
            "initial": 7,
            "operations": [("multiply", 6), ("add", 8), ("subtract", 10), ("divide", 5)],
            "requested_outputs": REQUESTED_OUTPUTS,
        },
        {
            "case_id": "math_04",
            "prompt": """Empieza con 81, divide entre 9, suma 11, multiplica por 2 y resta 10.
Dime:
- procedimiento
- resultado final""".strip(),
            "initial": 81,
            "operations": [("divide", 9), ("add", 11), ("multiply", 2), ("subtract", 10)],
            "requested_outputs": REQUESTED_OUTPUTS,
        },
        {
            "case_id": "math_05",
            "prompt": """Empieza con 3, suma 17, multiplica por 2, resta 5, divide entre 5 y suma 1.
Dime:
- procedimiento
- resultado final""".strip(),
            "initial": 3,
            "operations": [("add", 17), ("multiply", 2), ("subtract", 5), ("divide", 5), ("add", 1)],
            "requested_outputs": REQUESTED_OUTPUTS,
        },
    ]


def render_number(value: int | float) -> int | float:
    return int(value) if isinstance(value, float) and value.is_integer() else value


def apply_operation(value: int | float, operation: str, amount: int | float) -> tuple[int | float, str]:
    previous = value
    if operation == "add":
        value = value + amount
        symbol = "+"
    elif operation == "subtract":
        value = value - amount
        symbol = "-"
    elif operation == "multiply":
        value = value * amount
        symbol = "×"
    elif operation == "divide":
        if amount == 0:
            raise ValueError("No se puede dividir entre cero.")
        value = value / amount
        symbol = "÷"
    else:
        raise ValueError(f"Operación no soportada: {operation!r}")
    value = render_number(value)
    return value, f"{render_number(previous)} {symbol} {render_number(amount)} = {value}"


def execute_operations(initial: int | float, operations: list[tuple[str, int | float]]) -> dict[str, Any]:
    value = initial
    procedure = []
    for operation, amount in operations:
        value, explanation = apply_operation(value, operation, amount)
        procedure.append(explanation)
    return {"procedimiento": procedure, "resultado_final": value}


cases = make_math_batch()
lab.show({
    "case_count": len(cases),
    "primer_prompt": cases[0]["prompt"],
    "primer_caso": cases[0],
    "all_cases": cases,
})


## 2) Definir `transition_fn` y `reward_fn`

El environment no sabe si detrás hay tool, agent, multi-agent, LangGraph u OpenAI Agents. Sólo ejecuta una transición por fila y calcula reward.


In [ ]:
def arithmetic_transition(row: dict[str, Any], action: Any, info: dict[str, Any]) -> dict[str, Any]:
    """Environment transition function independiente de agentes o graphs."""

    answer = execute_operations(row["initial"], row["operations"])
    previous_memory = info.get("memory") or {}
    return {
        "selected_tool": "environment_transition",
        "summary": f"{row['case_id']}: result={answer['resultado_final']}",
        "prompt": row["prompt"],
        "procedure": answer["procedimiento"],
        "result": answer["resultado_final"],
        "requested_outputs": row["requested_outputs"],
        "respuesta_materializada": {key: answer[key] for key in row["requested_outputs"] if key in answer},
        "ok": True,
        "memory": {
            **previous_memory,
            "processed": [*previous_memory.get("processed", []), row["case_id"]],
            "last_result": answer["resultado_final"],
        },
    }


def arithmetic_reward(state: dict[str, Any], row: dict[str, Any], action: Any, env: lab.AgenticEnvironment) -> float:
    """Reward de 1 punto cuando la transición terminó correctamente."""

    return 1.0 if state.get("ok") else 0.0

lab.show({
    "transition_fn": "row.initial + row.operations -> respuesta_materializada",
    "reward_fn": "state.ok == True",
})


## 3) Construir y ejecutar Environment

`AgenticEnvironment` recorre records como episodio: `reset()` inicializa, `step()` avanza y `summary()` resume.


In [ ]:
env = lab.AgenticEnvironment(
    records=cases,
    name="fundamentals_environment_eval",
    transition_fn=arithmetic_transition,
    reward_fn=arithmetic_reward,
    render_mode="history",
)

observation, info = env.reset(seed=247)

while observation is not None:
    observation, reward, terminated, truncated, info = env.step()
    if terminated or truncated:
        break

lab.show(env.summary())


## 4) Finalize del episodio

En environment/evals, `finalize` no crea una respuesta conversacional. Cierra el episodio: toma el historial de steps y materializa un resumen auditable para usuario/evaluador.

In [ ]:
def finalize_episode(environment: lab.AgenticEnvironment) -> dict[str, Any]:
    cases = []
    for event in environment.history:
        graph_state = event.graph_state
        cases.append({
            "case_id": event.row.get("case_id"),
            "prompt": event.row.get("prompt"),
            "resultado_final": graph_state.get("result"),
            "procedimiento": graph_state.get("procedure"),
            "reward": event.reward,
            "ok": bool(graph_state.get("ok")),
        })

    summary = environment.summary()
    return {
        "requested_outputs": REQUESTED_OUTPUTS,
        "steps": summary["steps"],
        "passed_steps": summary["passed_steps"],
        "failed_steps": summary["failed_steps"],
        "total_reward": summary["total_reward"],
        "cases": cases,
    }


episode_final = finalize_episode(env)
lab.show(episode_final, title="Finalize · episode outputs")

## 5) Lineage Memory del episodio

Aquí se ve “qué pasó” paso a paso: decisiones, reward y evidencia mínima.


In [ ]:
lineage = env.lineage(
    question="Procesa 5 casos aritméticos multi-operación; el primero es el problema default de fundamentals.",
    goal="Explicar el episodio environment/eval sin depender de un framework externo.",
    tags=["fundamentals", "environment", "eval"],
)

lab.show(lineage)
lab.show({"compact_context": lineage.to_prompt_context(max_chars=1200)})


## 6) Definir tools, contrato, policy y agente de eval

`run_eval` eval?a un agente sobre casos declarativos. Para mantenerlo local y determinista, este agente usa `python-direct`.

Lo profesional aqu? es declarar contrato y policy arriba; despu?s `lab.agent(...)` solo recibe objetos ya nombrados.

In [ ]:
@lab.tool
def add(a: float, b: float) -> dict:
    return {"value": a + b}


@lab.tool
def subtract(a: float, b: float) -> dict:
    return {"value": a - b}


@lab.tool
def multiply(a: float, b: float) -> dict:
    return {"value": a * b}


@lab.tool
def divide(a: float, b: float) -> dict:
    return {"value": a / b}


eval_tools = [add, subtract, multiply, divide]

eval_contract = lab.AgentContract(
    tool_expectation=lab.expect.any_of("add", "subtract", "multiply", "divide"),
)

eval_policy = lab.RunPolicy(
    max_tool_calls=1,
    temperature=0.0,
    trace="compact",
)

eval_agent = lab.agent(
    name="fundamentals_eval_agent",
    instructions="Ejecuta una operaci?n estructurada usando las tools disponibles.",
    tools=eval_tools,
    engine="python-direct",
    contract=eval_contract,
    policy=eval_policy,
)

lab.show({
    "eval_agent": eval_agent.info(),
    "eval_tools": [tool.name for tool in eval_tools],
    "eval_contract": eval_contract.model_dump(mode="json"),
    "eval_policy": eval_policy.model_dump(mode="json"),
})

## 7) Construir casos de eval desde el mismo batch

Para mantener simple el ejemplo, cada caso de eval valida la **primera operación** de cada problema. El environment ya valida el problema completo multi-operación.


In [ ]:
eval_cases = []
for row in cases:
    first_op, amount = row["operations"][0]
    expected_first = execute_operations(row["initial"], [(first_op, amount)])["resultado_final"]
    eval_cases.append({
        "name": row["case_id"],
        "input": {"tool": first_op, "input": {"a": row["initial"], "b": amount}},
        "expected": {
            "must_call": [first_op],
            "data_contains": {"tool": first_op, "ok": True, "value": expected_first},
        },
    })

lab.show({
    "eval_cases": eval_cases,
    "nota": "Los expected de eval se calculan desde la primera operación declarada en cada record.",
})


## 8) Ejecutar `run_eval` y proyectar Lineage Memory

`run_eval` usa el patrón batch → step → validación → score. El reporte también puede producir lineage.


In [ ]:
report = lab.run_eval(eval_agent, eval_cases)
lab.show(report.to_dict())

report_lineage = report.lineage(
    name="fundamentals.arithmetic.eval.lineage",
    question="Evalúa 5 decisiones aritméticas con python-direct.",
    goal="Mostrar que evals y environments comparten batch, validación y lineage.",
)
lab.show(report_lineage)

lab.human_result(
    title="Eval batch + Lineage Memory · fundamentals",
    result=report,
    pretty=True,
    show_lineage=True,
    lineage=report_lineage,
)


## Lo importante

- Environment procesa episodios por step.
- `finalize_episode(...)` cierra el episodio desde `env.history`, no desde un texto libre.
- Eval reutiliza el patrón batch, pero aquí validamos un subproblema para mantener el demo corto.
- Lineage Memory compacta lo ocurrido sin reenviar todo el historial bruto.
- El problema default queda incluido como `math_01_default`.

## Coverage API de este notebook

Esta tabla deja explícito qué parte de Agentic Systems queda materializada aquí.


In [ ]:
api_coverage = [
    {
        "api": "AgenticEnvironment",
        "description": "Materializa un entorno con transicion, reward y episodios."
    },
    {
        "api": "records",
        "description": "Declara los casos del entorno como datos, no como narrativa oculta."
    },
    {
        "api": "transition_fn",
        "description": "Define como cambia el estado paso a paso."
    },
    {
        "api": "reward_fn",
        "description": "Materializa esta parte de la API con un ejemplo directo y verificable."
    },
    {
        "api": "env.reset",
        "description": "Materializa esta parte de la API con un ejemplo directo y verificable."
    },
    {
        "api": "env.step",
        "description": "Materializa esta parte de la API con un ejemplo directo y verificable."
    },
    {
        "api": "env.summary",
        "description": "Materializa esta parte de la API con un ejemplo directo y verificable."
    },
    {
        "api": "finalize_episode",
        "description": "Materializa esta parte de la API con un ejemplo directo y verificable."
    },
    {
        "api": "env.lineage",
        "description": "Materializa esta parte de la API con un ejemplo directo y verificable."
    },
    {
        "api": "@lab.tool",
        "description": "Construye herramientas declarativas para reutilizarlas en agentes y grafos."
    },
    {
        "api": "lab.agent(engine='python-direct')",
        "description": "Materializa esta parte de la API con un ejemplo directo y verificable."
    },
    {
        "api": "run_eval",
        "description": "Ejecuta evals sobre el mismo dominio que el environment."
    },
    {
        "api": "EvalReport.lineage",
        "description": "Materializa esta parte de la API con un ejemplo directo y verificable."
    },
    {
        "api": "lineage.to_prompt_context",
        "description": "Materializa esta parte de la API con un ejemplo directo y verificable."
    },
    {
        "api": "5-case arithmetic batch",
        "description": "Materializa esta parte de la API con un ejemplo directo y verificable."
    },
    {
        "api": "default case included",
        "description": "Materializa esta parte de la API con un ejemplo directo y verificable."
    }
]

lab.show({"notebook": "10_environment_eval_api.ipynb", "api_coverage": api_coverage})